# 3D Conformational Descriptors of CycPeptMPDB — UMAP Analysis

Cyclic peptides can cross membranes despite high polarity by folding into compact,
H-bond-shielded conformations in lipophilic environments and re-exposing polar groups in
water (*chameleonic* behavior). 2D descriptors are blind to this. Here we derive **3D
ensemble ΔPSA** descriptors and use UMAP to map how they organize CycPeptMPDB relative to
measured PAMPA permeability.

This notebook is self-contained: it lists the data and scripts used, regenerates each
figure, and states which data subset each figure is built from.

## Data

| | |
|---|---|
| **Source** | CycPeptMPDB v1.2 — 8,466 cyclic peptides with experimental permeability |
| **PAMPA subset** | 7,298 compounds with a PAMPA LogPexp value (1,168 without were dropped) |
| **Clean subset** | 1,566 from the two internally consistent PAMPA protocols — Furukawa 2016 + Chugai 2013 |
| **Label** | permeable = PAMPA LogPexp ≥ −6.0 log cm/s |
| **Feature matrix** | `results/archive/feature_matrix.csv` (all 2D + 3D + DB features, all 7,298 compounds) |

All descriptors were computed once for all 7,298 compounds; every figure below is a *view*
of that one matrix (a subset chosen to match the question), not a separate run.

### Descriptor groups

| Group | Features | Origin |
|---|---|---|
| 2D baseline | MolLogP, TPSA, MolWt, NumHDonors, NumHAcceptors, NumRotatableBonds, RingCount | RDKit (conformation-blind) |
| 3D Δ (Tier-1) | delta_psa3d, psa3d_std, psa3d_spread, delta_hb, delta_Rg, delta_NPR1/2, delta_psa3d_per_mw | vacuum ETKDGv3 + MMFF94s ensemble |
| DB 3D (control) | H2O_3DPSA, CHCl3_3DPSA, delta_3DPSA_db | CycPeptMPDB single-structure values |

**Note on the Tier-1 ΔPSA.** `delta_psa3d` = PSA(highest-PSA conformer) − PSA(lowest-PSA
conformer) over a *vacuum* ensemble. It measures a molecule's **accessible PSA range**
(capacity to switch), not the Boltzmann-weighted populations it actually holds in solvent —
so it is exploratory. Implicit-solvent ensembles (CREST/xTB) are the physics-grounded next
step. Read the maps below as an exploratory picture of chemical space.

## Setup

In [ ]:
import os, sys, glob, subprocess
from pathlib import Path
from IPython.display import Image, display

# run from the repo root (folder containing 'scripts' and 'results')
while not (Path.cwd() / 'scripts' / 'ml' / 'umap_visualization.py').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
ROOT = Path.cwd(); print('repo root:', ROOT)

MATRIX = 'results/archive/feature_matrix.csv'   # the one feature matrix all figures draw from
FIGDIR = Path('results/figures')
os.environ['MPLBACKEND'] = 'Agg'

REGENERATE = False   # figures are pre-built; set True to rebuild them from the matrix (slow: UMAP)

def generate(*args):
    """Regenerate a figure via a repo script. Skipped unless REGENERATE=True, so the notebook
    renders instantly from the committed figures; flip the flag to reproduce from scratch."""
    if not REGENERATE:
        return
    print('>', ' '.join(map(str, args)))
    subprocess.run([sys.executable, *map(str, args)], check=False)

def show(prefix, which='latest'):
    """Display a figure. `which`: 'latest' (default), 'big' or 'small' (by compound count in
    the filename, to pick full-dataset vs clean-subset panels that share a prefix)."""
    files = list(FIGDIR.glob(f'{prefix}*.png'))
    if not files:
        print('(missing — set REGENERATE=True and re-run, or run scripts in the Reproduce cell)'); return
    def n_of(p):
        try: return int(p.stem.split('_umap_')[1].split('_')[0])
        except Exception: return 0
    files.sort(key=lambda p: (n_of(p), p.stat().st_mtime))
    pick = {'big': files[-1], 'small': files[0]}.get(which, max(files, key=lambda p: p.stat().st_mtime))
    display(Image(str(pick)))

## Figure 0 — Single-feature AUC by descriptor

*Data: clean subset (Furukawa + Chugai, n = 1,566, blue) vs. full dataset (n = 7,298, orange).*

AUC-ROC of each descriptor against PAMPA permeability (threshold −6.0 log cm/s). On the clean
subset, ensemble ΔPSA (0.69) and per-MW-normalized ΔPSA (0.59) carry signal that **collapses
on the full dataset** (0.51 / 0.47); the DB single-structure ΔPSA stays at chance in both.
MolLogP *inverts* on the clean subset (0.32, i.e. 0.68 flipped) — the large polar macrolides
dominate the permeable population, reversing the usual lipophilicity/permeability relationship.

In [ ]:
generate('scripts/ml/auc_by_subset.py', '--matrix', MATRIX, '--out', 'results/figures/auc_by_subset.png')
show('auc_by_subset')

## Figure 1 — Panel A: 2D descriptors

*Data: full dataset, 6,938 compounds with complete 2D features.*

UMAP of the 2D topological features. Track A = K-Medoids archetypes, Track B = HDBSCAN,
Clincher = points colored by PAMPA after the blind embedding. The 2D descriptors fragment the
space and show no clean permeable/impermeable layout — Cyclosporin A (★) sits in the
undifferentiated core. *2D descriptors do not organize this space in a permeability-relevant way.*

In [ ]:
generate('scripts/ml/umap_visualization.py', '--matrix', MATRIX, '--outdir', 'results', '--panels', 'Panel_A_2D', '--k', '8')
show('Panel_A_2D_umap_')

## Figure 2 — Panel B: 3D Δ descriptors (full dataset)

*Data: full dataset, 6,937 compounds with complete 3D features.*

Same embedding on the 3D Δ features. The space forms a few broad regions rather than 2D's many
fragments, but the PAMPA coloring (Clincher) is mixed — on the pooled multi-source labels the
permeability signal is washed out. *The 3D descriptors find broad structure even where the
heterogeneous labels no longer track it.*

In [ ]:
generate('scripts/ml/umap_visualization.py', '--matrix', MATRIX, '--outdir', 'results', '--panels', 'Panel_B_3D_delta', '--k', '8')
show('Panel_B_3D_delta_umap_', which='big')

## Figure 3 — Panel B: 3D Δ descriptors (clean subset)

*Data: clean subset (Furukawa + Chugai, ~1,502 compounds).*

The same 3D Δ embedding on the clean subset. Now the layout separates **impermeable (red,
left)** from **permeable (green, right)**, with Cyclosporin A (★) in the permeable lobe —
the embedding is blind to PAMPA, so the alignment is emergent. *On clean labels, the 3D Δ
layout tracks permeability.*

In [ ]:
generate('scripts/ml/umap_visualization.py', '--matrix', MATRIX, '--outdir', 'results', '--sources', '2016_Furukawa', '2013_CHUGAI', '--panels', 'Panel_B_3D_delta', '--k', '8')
show('Panel_B_3D_delta_umap_', which='small')

## Figure 4 — Panel C: combined 2D + 3D, with size axes

*Data: clean subset (Furukawa + Chugai, n = 1,566).*

Combined 2D+3D embedding with two extra colorings: **Track D = Molecular Weight** and
**Track E = residue count** (≥9-residue compounds ringed). The permeable lobe is the
**high-MW, ≥9-residue macrolides** (median MW 1,180 vs 820 Da for the impermeable lobe). The
two populations that appear throughout are essentially a **size distinction** — large
chameleonic macrolides vs. smaller cyclic peptides.

In [ ]:
generate('scripts/ml/umap_visualization.py', '--matrix', MATRIX, '--outdir', 'results', '--sources', '2016_Furukawa', '2013_CHUGAI', '--panels', 'Panel_C_combined', '--k', '8')
show('Panel_C_combined_umap_')

## Reproduce

```bash
# environment (any Linux/Mac/WSL/Colab, or a local conda env)
pip install pandas numpy matplotlib scikit-learn umap-learn hdbscan scikit-learn-extra

# each figure (data = results/archive/feature_matrix.csv)
python scripts/ml/auc_by_subset.py       --matrix results/archive/feature_matrix.csv --out results/figures/auc_by_subset.png
python scripts/ml/umap_visualization.py  --matrix results/archive/feature_matrix.csv --outdir results --panels Panel_A_2D Panel_B_3D_delta --k 8
python scripts/ml/umap_visualization.py  --matrix results/archive/feature_matrix.csv --outdir results --sources 2016_Furukawa 2013_CHUGAI --panels Panel_B_3D_delta Panel_C_combined --k 8
```

**Scripts:** `scripts/ml/auc_by_subset.py`, `scripts/ml/umap_visualization.py`.
**Data:** `results/archive/feature_matrix.csv`.

UMAP is used here as a **visual argument** — a map of how the descriptors organize chemical
space — not as a statistical clustering claim; cluster-validity metrics are omitted from the
panels accordingly. The quantitative results are the AUC comparison (Figure 0) and the MW /
residue-size split (Figure 4).